In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import warnings 
warnings.filterwarnings('ignore') 



In [6]:
df = pd.read_csv('../data//cleaned__data.csv')
df.head(1)
df.isnull().sum()

Unnamed: 0           0
reviewID             0
reviewerID           0
restaurantID         0
date                 0
rating               0
reviewUsefulCount    0
reviewContent        0
flagged              0
name                 0
location             0
yelpJoinDate         0
friendCount          0
reviewCount          0
firstCount           0
usefulCount          0
coolCount            0
funnyCount           0
complimentCount      0
tipCount             0
fanCount             0
restaurantRating     0
ReviewLength         0
review_year          0
dtype: int64

#
# Feature Engineering
#

## Sentiment Score and Sentiment 

In [11]:
!pip install textblob


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from textblob import TextBlob

def get_sentiment(text):
    # If the text is missing, return exactly neutral (0.0)
    if pd.isna(text):
        return 0.0
    return TextBlob(str(text)).sentiment.polarity

df['sentiment_score'] = df['reviewContent'].apply(get_sentiment)


sentiment_std = df.groupby('reviewerID')['sentiment_score'].std().reset_index()
sentiment_std.rename(columns={'sentiment_score': 'reviewer_sentiment_var'}, inplace=True)
df = df.merge(sentiment_std, on='reviewerID', how='left')

df['reviewer_sentiment_var'] = df['reviewer_sentiment_var'].fillna(0)



display(df[['reviewerID', 'reviewContent', 'sentiment_score', 'reviewer_sentiment_var']].head())

Sentiment Score calculated.
Sentiment Variance calculated and merged.


,reviewerID,reviewContent,sentiment_score,reviewer_sentiment_var
0,hfQu0YNy_XW5oiiripgUFg,sunda amazing i heard many good things finally...,0.318651,0.000000
1,_jsZl-USMgrVVasNg50wAQ,absolutely fantastic foodie community table gr...,0.215714,0.000000
2,sZxXpvmBUN2fSCtK_BZFoQ,i work right rarely go here they 5 personal st...,0.237395,0.057734
3,YMS9Fzy0OcOXFcS_qms_pg,this best big 3 brazilian steakhouses chicago ...,0.500000,0.000000
4,uvWWKett-BIRbvyLZXaloQ,i lunch the gage group 8 this first time there...,0.224152,0.000000


## Lexical Diverstiy

In [13]:
# Lexical Diversity: Type-Token Ratio (TTR)
# The Concept:
# To understand TTR, you just need to know two linguistic terms:

# Tokens: The total number of words in a text.

# Types: The number of unique words in a text.

# The formula is simply: TTR = Types / Tokens (Unique Words divided by Total Words).

# The Range (0.0 to 1.0):

# A score of 1.0 means every single word in the review is unique (highest diversity).

# A score closer to 0.0 means the text is highly repetitive, reusing the same words over and over (lowest diversity).

# Why it catches fakes:
# Genuine humans naturally use a wide, dynamic vocabulary to describe an experience. 
# Paid review farmers, however, are usually writing hundreds of reviews a day.
# To save time (and brainpower), they rely on templates, repetitive sentence structures,
# and keyword stuffing (e.g., repeating the restaurant's name and the word "good").

In [14]:
import pandas as pd

print("Starting Lexical Diversity (TTR) Calculation...")

# Define the simplest function to calculate TTR
def get_ttr(text):
    # Handle missing text
    if pd.isna(text):
        return 0.0
    
    # 1. Convert to lowercase and split by spaces into a list of words
    words = str(text).lower().split()
    
    # 2. Count total words (Tokens)
    total_words = len(words)
    
    # Avoid dividing by zero if the review is somehow completely empty
    if total_words == 0:
        return 0.0
    
    # 3. Count unique words (Types) by turning the list into a set
    unique_words = len(set(words))
    
    return unique_words / total_words

df['lexical_diversity_ttr'] = df['reviewContent'].apply(get_ttr)


display(df[['reviewContent', 'ReviewLength', 'lexical_diversity_ttr']].head())

Starting Lexical Diversity (TTR) Calculation...


,reviewContent,ReviewLength,lexical_diversity_ttr
0,sunda amazing i heard many good things finally...,72,0.847222
1,absolutely fantastic foodie community table gr...,12,1.000000
2,i work right rarely go here they 5 personal st...,102,0.686275
3,this best big 3 brazilian steakhouses chicago ...,30,0.933333
4,i lunch the gage group 8 this first time there...,93,0.860215


In [15]:
# Word Repetitiveness (Redundancy Score)
# The Concept:
# Word Repetitiveness calculates the exact percentage of a text that consists of duplicated words. We find this by subtracting the unique words from the total words to see how many "redundant" words are left, and then divide that by the total word count.

# The Range (0.0 to 1.0):

# A score of 0.0 means there is 0% repetition (every single word is completely unique).

# A score closer to 1.0 means the review is highly redundant, consisting mostly of the exact same words looped over and over.

# Why it catches fakes:
# Just like TTR, this targets the laziness of bot networks. A bot might try to artificially increase its ReviewLength (since very short reviews get flagged by Yelp filters) by simply repeating keywords. This artificially inflates the word count without adding any actual information.

# Genuine Example: "The steak was perfect, but the service was slow."

# Total Words: 9

# Unique Words: 8 (only "the" is repeated)

# Redundant Words: 1

# Repetitiveness: 1 / 9 = 0.11 (11%)

# Bot Example: "Very good food. Good place. Good good good."

# Total Words: 8

# Unique Words: 4 ("very", "good", "food", "place")

# Redundant Words: 4

# Repetitiveness: 4 / 8 = 0.50 (50%)

In [17]:

# Define the function to calculate redundancy
def get_repetitiveness(text):
    if pd.isna(text):
        return 0.0
    
    #  Convert to lowercase and split into words
    words = str(text).lower().split()
    total_words = len(words)
    
    # Avoid dividing by zero
    if total_words == 0:
        return 0.0
    
    # Count unique words
    unique_words = len(set(words))
    
    #  Calculate how many words are exact duplicates
    duplicate_words = total_words - unique_words
    
    return duplicate_words / total_words


df['word_repetitiveness'] = df['reviewContent'].apply(get_repetitiveness)


display(df[['reviewContent', 'lexical_diversity_ttr', 'word_repetitiveness']].head())

,reviewContent,lexical_diversity_ttr,word_repetitiveness
0,sunda amazing i heard many good things finally...,0.847222,0.152778
1,absolutely fantastic foodie community table gr...,1.000000,0.000000
2,i work right rarely go here they 5 personal st...,0.686275,0.313725
3,this best big 3 brazilian steakhouses chicago ...,0.933333,0.066667
4,i lunch the gage group 8 this first time there...,0.860215,0.139785


In [18]:
# The Concept:
# This metric calculates the percentage of words in a review that are typed entirely in uppercase letters (e.g., "HORRIBLE", "AMAZING"). We count the number of all-caps words and divide it by the total word count.

# The Range (0.0 to 1.0):

# A score of 0.0 means normal casing (or all lowercase).

# A score closer to 1.0 means the reviewer is "screaming" the entire time.

# Why it catches fakes:
# Genuine humans use capitalization sparingly for emphasis. Paid bots, however, are designed to manipulate the reader's attention.
# If they are paid to destroy a competitor, they scream. If they are paid to boost a client, they scream. Extreme capitalization is a major hallmark of non-genuine, emotionally manipulative text.

# Genuine Example: "The food was great, but the music was a bit LOUD."

# Total Words: 10

# All-Caps Words: 1 ("LOUD")

# Ratio: 1 / 10 = 0.10 (10%)

# Bot Example: "BEST FOOD EVER YOU MUST GO NOW AMAZING!!!"

# Total Words: 8

# All-Caps Words: 8

# Ratio: 8 / 8 = 1.0 (100%)

In [19]:

def get_caps_ratio(text):
    if pd.isna(text):
        return 0.0
    
    # Split the original text (NOT lowercased this time) into words
    words = str(text).split()
    total_words = len(words)
    
    # Avoid dividing by zero
    if total_words == 0:
        return 0.0
    
    # Count how many words are entirely uppercase
    # .isupper() ensures that things like "123" or "..." don't get counted as uppercase
    caps_words = sum(1 for word in words if word.isupper())
    
    # Return the ratio
    return caps_words / total_words


df['capitalization_ratio'] = df['reviewContent'].apply(get_caps_ratio)


display(df[['reviewContent', 'sentiment_score', 'lexical_diversity_ttr', 'capitalization_ratio']].head())

,reviewContent,sentiment_score,lexical_diversity_ttr,capitalization_ratio
0,sunda amazing i heard many good things finally...,0.318651,0.847222,0.0
1,absolutely fantastic foodie community table gr...,0.215714,1.000000,0.0
2,i work right rarely go here they 5 personal st...,0.237395,0.686275,0.0
3,this best big 3 brazilian steakhouses chicago ...,0.500000,0.933333,0.0
4,i lunch the gage group 8 this first time there...,0.224152,0.860215,0.0


In [20]:
# A genuine human has a physical body. They live in one city, eat mostly in that city, and occasionally travel for vacation (resulting in low to moderate entropy).
# A paid bot network operates in cyberspace. A bot farm in a single day might be paid to 5-star a pizza shop in New York, 1-star a mechanic in Los Angeles, and 5-star a dentist in Miami. A bot's geographic footprint is physically impossible, resulting in high location entropy.

# Genuine Example: User writes 18 reviews in "Chicago, IL" and 2 reviews in "Orlando, FL" (Vacation).

# Entropy Score: Very Low (~0.3)

# Bot Example: User writes 20 reviews across 20 completely different US cities in the span of a month.

# Entropy Score: Very High (~3.0)

In [23]:
import pandas as pd
import numpy as np
from scipy.stats import entropy


#  Define the function to calculate Shannon Entropy for a user's locations
def get_location_entropy(location_series):
    # Count how many times the user reviewed in each specific city
    counts = location_series.value_counts()
    
    # Convert those counts into probabilities (e.g., 80% Chicago, 20% Miami)
    probabilities = counts / counts.sum()
    
    # Calculate the entropy of that distribution
    # (Base 'e' is default, which is standard for this metric)
    return entropy(probabilities)

#  Group by the reviewer and apply the function to their 'location' history
user_entropy = df.groupby('reviewerID')['location'].apply(get_location_entropy).reset_index()

#  Rename the column for clarity
user_entropy.rename(columns={'location': 'location_entropy'}, inplace=True)

# Merge this new feature back into our main dataframe
df = df.merge(user_entropy, on='reviewerID', how='left')

# Handle users with only 1 review
# If a user only has 1 review total, they have 0 geographic scatter. 
df['location_entropy'] = df['location_entropy'].fillna(0)


display(df[['reviewerID', 'location', 'reviewCount', 'location_entropy']].head(10))

,reviewerID,location,reviewCount,location_entropy
0,hfQu0YNy_XW5oiiripgUFg,"Chicago, IL",1,0.0
1,_jsZl-USMgrVVasNg50wAQ,"Davenport, FL",1,0.0
2,sZxXpvmBUN2fSCtK_BZFoQ,"Chicago, IL",845,0.0
3,YMS9Fzy0OcOXFcS_qms_pg,"Chicago, IL",2,0.0
4,uvWWKett-BIRbvyLZXaloQ,"Stoneham, MA",1,0.0
5,-ZnHk8rEBKMDccy-Busuyw,"Oak Brook, IL",3,0.0
6,fJoVX8iLtCpdx0xI5TSXSA,"Cook, IL",42,0.0
7,FtFmC_cYO8NsCItbgEb9sg,"Bloomington, IL",201,0.0
8,Kh6rQmwr11UxDcE1xubxMA,"Chicago, IL",2,0.0
9,ax2VUpuWlDScCuvjPPhekQ,"Chicago, IL",13,0.0


In [24]:
# Genuine humans write reviews sporadically—usually after a notable dinner out, a vacation, or a particularly bad experience. Their velocity is extremely low because living a normal life prevents them from reviewing businesses every day.
# Paid bots are built for sheer volume. If an account is created and posts 50 reviews within its first week to fulfill a paid contract, its velocity will be mathematically impossible for a human to replicate.

# Genuine Example: User joined Yelp 1,000 days ago and has written 20 reviews.

# Calculation: 20 / 1000 = 0.02 reviews/day

# Bot Example: User joined Yelp 5 days ago and has written 50 reviews.

# Calculation: 50 / 5 = 10.0 reviews/day

In [26]:

# 1. Calculate 'days_active' (the number of days between joining Yelp and writing the review)
# .dt.days converts the pandas time-difference into a simple integer
df['days_active'] = (df['date'] - df['yelpJoinDate']).dt.days

# 2. Prevent division by zero 
# If they reviewed on the exact day they joined (0 days active), we treat it as 1 day.
df['days_active'] = df['days_active'].clip(lower=1)

# 3. Calculate Velocity (Total Reviews / Days Active)
df['review_velocity'] = df['reviewCount'] / df['days_active']


# Let's inspect the results to see how fast these users are reviewing
display(df[['reviewerID', 'reviewCount', 'days_active', 'review_velocity']].head(10))

TypeError: unsupported operand type(s) for -: 'str' and 'str'